# W3Q5 — Customer Segments by Conversion Time
**Question:** Can we identify distinct user segments based on how quickly they convert from trial to paid? What characterises each segment?

**Audience:** Marketing & Product Teams  
**Data source:** `ANALYTICS.MARTS.MART_SUBSCRIPTION_FUNNEL`  
**SQL:** `sql/W3Q5_customer_segments_by_conversion_time.sql`

---

**Segment definitions (days trial → 3m subscription):**
| Segment | Definition |
|---------|-----------|
| `fast` | ≤ 7 days — converted within or immediately after trial |
| `medium` | 8–30 days — converted within a month |
| `slow` | > 30 days — took more than a month |
| `never` | Started trial but never converted to 3m |

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from src.connection import query

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130})
PALETTE = sns.color_palette('muted', 8)

## Load Data

In [ ]:
with open('../sql/W3Q5_customer_segments_by_conversion_time.sql') as f:
    sql = f.read()

df = query(sql)
print(f"Rows: {len(df):,}  |  Segments: {df['conversion_speed'].value_counts().to_dict()}")
df.head(3)

## Segment Size & 12m Upsell Rate

In [ ]:
seg_order = ['fast', 'medium', 'slow', 'never']
seg = (
    df.groupby('conversion_speed')
      .agg(
          users=('user_id', 'count'),
          pct_with_12m=('has_12m_subscription', lambda x: round(x.mean() * 100, 1)),
          median_days_trial_to_3m=('days_trial_to_3m', 'median'),
          median_days_3m_to_12m=('days_3m_to_12m', 'median'),
          pct_paid=('acquisition_type', lambda x: round((x == 'paid').mean() * 100, 1)),
      )
      .reindex(seg_order)
      .reset_index()
)
seg['pct_of_total'] = (seg['users'] / seg['users'].sum() * 100).round(1)
display(seg)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(seg['conversion_speed'], seg['users'], color=PALETTE[:4], edgecolor='white')
axes[0].set_title('Segment Size', fontsize=12)
axes[0].set_xlabel('Conversion Speed')
axes[0].set_ylabel('Number of Users')
sns.despine(ax=axes[0])

axes[1].bar(seg['conversion_speed'], seg['pct_with_12m'], color=PALETTE[:4], edgecolor='white')
axes[1].set_title('12m Upsell Rate by Segment', fontsize=12)
axes[1].set_xlabel('Conversion Speed')
axes[1].set_ylabel('% Who Upgraded to 12m')
sns.despine(ax=axes[1])

fig.suptitle('Customer Segments by Trial-to-3m Conversion Speed', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../output/W3Q5_conversion_speed_segments.png', bbox_inches='tight')
plt.show()

## Segment Profile — Acquisition Type & Registration

In [ ]:
profile = (
    df.groupby(['conversion_speed', 'acquisition_type'])
      .size()
      .unstack(fill_value=0)
      .reindex(seg_order)
)
profile_pct = profile.div(profile.sum(axis=1), axis=0) * 100
display(profile_pct.round(1))

## Findings

- **Segment distribution (fast / medium / slow / never):** ...
- **12m upsell rate by segment:** ...
- **Fast converters — profile:** ...
- **Never converters — profile:** ...
- **Acquisition type split across segments:** ...
- **Recommendation:** ...